# 01.6 Losses and Optimizers

This notebook starts answering two core questions in training:

1. how wrong is the model?
2. in which direction should parameters be updated?

These two questions are handled by:

- loss function
- optimizer

## Learning Goals

After this notebook, you should be able to:

1. Understand the role of the loss function.
2. Distinguish common losses for regression and classification.
3. Understand input-output requirements for `MSELoss` and `CrossEntropyLoss`.
4. Understand the optimizer's role in parameter updates.
5. Read the meaning of `zero_grad()`, `backward()`, and `step()`.
6. Prepare for the full training loop.

In [ ]:
import torch
import torch.nn as nn

## What Is a Loss Function?

A loss function turns the difference between predictions and targets into a number that can be optimized.

In general:

- the model is closer to the target
- the model is farther from the target

## `MSELoss` for Regression

`MSELoss` is a very common loss for regression tasks.

Intuition:

- compute prediction error
- square it
- then average

In [ ]:
pred = torch.tensor([[2.5], [0.0], [2.0], [8.0]], dtype=torch.float32)
target = torch.tensor([[3.0], [-0.5], [2.0], [7.0]], dtype=torch.float32)

mse_loss = nn.MSELoss()
loss = mse_loss(pred, target)
print("MSE loss =", loss.item())

In regression, a common habit is to keep `pred` and `target` shapes as consistent as possible.


In [ ]:
# Exercise 1
# Use MSELoss to compute the loss below.

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# Exercise 1 Reference Solution

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])
loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)
print(loss.item())

## `CrossEntropyLoss` for Classification

`CrossEntropyLoss` is very common in multiclass classification.

Its input requirements are especially important:

- raw scores, often called logits
- class indices, usually `long`

A common mistake:

- Do not manually apply `softmax` before passing to `CrossEntropyLoss` 
  Do not manually apply `softmax` before passing outputs to `CrossEntropyLoss`.

In [ ]:
logits = torch.tensor(
    [
        [2.0, 0.5, -1.0],
        [0.1, 0.2, 2.5],
        [1.5, 1.1, 0.3],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 2, 1], dtype=torch.long)

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(logits, targets)

print("logits.shape =", logits.shape)
print("targets.shape =", targets.shape)
print("CrossEntropy loss =", loss.item())

Shape requirements:

- `logits.shape == (batch_size, num_classes)`
- `targets.shape == (batch_size,)`

Targets are class indices, not one-hot vectors.


In [ ]:
# Exercise 2
# Use CrossEntropyLoss to compute the classification loss below.

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# Exercise 2 Reference Solution

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)
print(loss.item())

## Optimizer

The loss tells you how wrong the model is, and the optimizer tells you how to update the parameters.

Common optimizers:

- `SGD`
- `Adam`

You do not need the full formulas yet, but you should know they update parameters based on gradients.


In [ ]:
model = nn.Linear(2, 1)
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.01)

print("SGD optimizer / SGD optimizer:")
print(optimizer_sgd)
print()
print("Adam optimizer / Adam optimizer:")
print(optimizer_adam)

## 5. `zero_grad()`、`backward()`、`step()`
## `zero_grad()`, `backward()`, and `step()`

The three most central actions in training are:

1. clear old gradients
2. compute new gradients from the loss
3. update parameters using the gradients

In [ ]:
torch.manual_seed(0)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])

print("parameters before update / parameters before update:")
for name, param in model.named_parameters():
    print(name, param.data)

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print()
print("loss =", loss.item())
print()
print("parameters after update / parameters after update:")
for name, param in model.named_parameters():
    print(name, param.data)

This short block is already very close to a full training loop.


In [ ]:
# Exercise 3
# Use the model and data below to perform one parameter update.
# Fill in:
# 1. optimizer.zero_grad()
# 2. pred = model(x)
# 3. loss = loss_fn(pred, y)
# 4. loss.backward()
# 5. optimizer.step()

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

# TODO

In [ ]:
# Exercise 3 Reference Solution

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print("loss =", loss.item())
for name, param in model.named_parameters():
    print(name, param.data)

## Intuition for `SGD` vs `Adam`

For now, keep a practical intuition-level comparison:

- more basic and direct
- often easier to use and often converges faster with default settings

This is not an absolute rule, but it is very useful early on.


## Mapping Task Type, Output Layer, and Loss Function

This is a very important mental mapping table:

- regression:
  output dimension often 1
 Common loss `MSELoss`
- multiclass classification:
  output dimension usually equals number of classes
 Common loss `CrossEntropyLoss`

If this mapping is confused, training often fails immediately with shape or dtype errors.


In [ ]:
# Exercise 4
# Decide which loss function fits each scenario.
# 1. predict tomorrow's temperature
# 
# Answer in one sentence and explain why.

Exercise 4 Reference Answer

1. Predict tomorrow's temperature: regression, so `MSELoss` or `L1Loss` is a natural fit.
2. Choose one class among several exclusive classes: multiclass classification, so use `CrossEntropyLoss` with logits of shape `(batch, num_classes)` and integer class labels.
3. Predict several independent yes/no labels at once: multilabel classification, so use `BCEWithLogitsLoss` with one logit per label.

Loss choice depends on the target meaning and expected output shape.

## Summary

The core chain in this notebook is:

- model outputs
- the loss compares outputs with targets
- backpropagation produces gradients
- the optimizer updates parameters using those gradients

You should now be able to answer:

1. What tasks do `MSELoss` and `CrossEntropyLoss` suit?
2. Why are `CrossEntropyLoss` targets usually class indices instead of one-hot vectors?
3. What do `zero_grad()`, `backward()`, and `step()` each do?
4. Why does task type affect output-layer design and loss choice?

Suggested next step:

- The next natural step is the full training-loop notebook.